# Práctica: Parametrización de Modelos
## Parte 2 — Experimentación con Parámetros

**Objetivo:** Entender cómo `temperature`, `top_p`, `presence_penalty` y `frequency_penalty` modifican el comportamiento de `gpt-4o` en Azure AI Foundry, con análisis comparativo para cada parámetro.

---

### Estructura del notebook
1. Configuración del entorno
2. Experimentación con `temperature`
3. Experimentación con `top_p`
4. Experimentación con Penalties (`presence_penalty` y `frequency_penalty`)
5. Preguntas teóricas
6. Conclusiones

---
## 1. Configuración del entorno

In [1]:
%pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

AZURE_ENDPOINT    = os.getenv("AZURE_ENDPOINT")
AZURE_API_KEY     = os.getenv("AZURE_API_KEY")
AZURE_API_VERSION = os.getenv("AZURE_API_VERSION")
DEPLOYMENT_NAME   = os.getenv("DEPLOYMENT_NAME")

client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION,
)

def chat(
    prompt: str,
    temperature: float = 1.0,
    top_p: float = 1.0,
    presence_penalty: float = 0.0,
    frequency_penalty: float = 0.0,
    max_tokens: int = 400,
    system: str = "Eres un asistente útil."
) -> str:
    """Wrapper para llamadas al modelo con todos los parámetros configurables."""
    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        temperature=temperature,
        top_p=top_p,
        presence_penalty=presence_penalty,
        frequency_penalty=frequency_penalty,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


def mostrar_comparativa(resultados: dict, titulo: str):
    """Muestra resultados de forma visual y comparativa."""
    print(f"\n{'='*60}")
    print(f" {titulo}")
    print(f"{'='*60}")
    for config, respuesta in resultados.items():
        print(f"\n--- {config} ---")
        print(respuesta)
        print()

print("✅ Cliente Azure OpenAI configurado correctamente")

✅ Cliente Azure OpenAI configurado correctamente


---
## 2. Experimentación con `temperature`

### ¿Qué controla `temperature`?

La `temperature` controla la **aleatoriedad** en la selección de tokens. Técnicamente, escala los logits antes de la distribución softmax:
- **temperature = 0**: Siempre elige el token más probable (determinista).
- **temperature = 1**: Distribución de probabilidades sin modificar.
- **temperature > 1**: Aplana la distribución → más aleatoriedad → respuestas más inesperadas.

### Experimento: generación de eslóganes publicitarios

Usamos una tarea creativa para ver cómo cambia la variedad y creatividad de las respuestas.

In [9]:
PROMPT_ESLOGANES = "Genera 3 eslóganes publicitarios originales para una marca de café artesanal llamada 'Altura'. Sé creativo."

temperaturas = [0.0, 0.5, 1.0]
resultados_temp = {}

for temp in temperaturas:
    # Ejecutar 2 veces con temp=0 para mostrar determinismo
    respuesta = chat(PROMPT_ESLOGANES, temperature=temp, top_p=1.0)
    resultados_temp[f"temperature={temp}"] = respuesta

mostrar_comparativa(resultados_temp, "Efecto de temperature en generación creativa")


 Efecto de temperature en generación creativa

--- temperature=0.0 ---
1. **"Altura: Donde cada sorbo te eleva al sabor más puro."**  
2. **"Café Altura: Cultivado en las nubes, disfrutado en la cima."**  
3. **"Altura: Artesanía en cada grano, pasión en cada taza."**  


--- temperature=0.5 ---
1. **"Altura: Donde cada sorbo te eleva al sabor más puro."**

2. **"Altura: Café que nace en las cumbres, para inspirar tus días."**

3. **"Altura: Artesanía en cada grano, pasión en cada taza."**


--- temperature=1.0 ---
1. **"Altura: Tan alto como tus sueños, tan cercano como cada sorbo."**  
2. **"Altura: Café que eleva tus sentidos, un grano a la vez."**  
3. **"Altura: Donde el arte del café toca el cielo."**  



In [4]:
# Demostración de DETERMINISMO con temperature=0
# Si ejecutas esta celda varias veces, la respuesta debería ser idéntica o muy similar
print("Ejecución 1 con temperature=0.0:")
print(chat(PROMPT_ESLOGANES, temperature=0.0))
print("\nEjecución 2 con temperature=0.0:")
print(chat(PROMPT_ESLOGANES, temperature=0.0))

Ejecución 1 con temperature=0.0:
1. **"Altura: Donde cada sorbo te eleva al sabor más puro."**  
2. **"Altura: Café que nace en las cumbres, para conquistar tus sentidos."**  
3. **"Altura: Artesanía en cada grano, pasión en cada taza."**  

Ejecución 2 con temperature=0.0:
1. **"Altura: Donde cada sorbo te eleva al sabor más puro."**  
2. **"Altura: Café que nace en las cimas, para conquistar tus sentidos."**  
3. **"Altura: Artesanía en cada grano, pasión en cada taza."**  


### Experimento 2: tarea técnica (código)

Comprobamos si `temperature` afecta igual a tareas técnicas donde hay una respuesta "correcta".

In [10]:
PROMPT_CODIGO = "Escribe una función Python que compruebe si una cadena de texto es un palíndromo. Solo el código, sin explicaciones."

resultados_codigo = {}
for temp in [0.0, 0.7, 1]:
    resultados_codigo[f"temperature={temp}"] = chat(PROMPT_CODIGO, temperature=temp, top_p=1.0)

mostrar_comparativa(resultados_codigo, "Efecto de temperature en código Python")


 Efecto de temperature en código Python

--- temperature=0.0 ---
```python
def es_palindromo(cadena):
    cadena = ''.join(c for c in cadena.lower() if c.isalnum())
    return cadena == cadena[::-1]
```


--- temperature=0.7 ---
```python
def es_palindromo(cadena):
    cadena = ''.join(c for c in cadena.lower() if c.isalnum())
    return cadena == cadena[::-1]
```


--- temperature=1 ---
```python
def es_palindromo(cadena):
    cadena = ''.join(c for c in cadena.lower() if c.isalnum())
    return cadena == cadena[::-1]
```



---
## 3. Experimentación con `top_p`

### ¿Qué controla `top_p`?

`top_p` (también llamado **nucleus sampling**) restringe el conjunto de tokens candidatos a los que, sumados en orden de probabilidad, alcanzan la probabilidad acumulada `p`:
- **top_p = 0.1**: Solo considera el 10% de masa de probabilidad → muy pocos tokens candidatos → respuestas conservadoras.
- **top_p = 0.9**: Considera el 90% de masa de probabilidad → más tokens candidatos → respuestas más variadas.
- **top_p = 1.0**: Todos los tokens son candidatos (comportamiento por defecto).

**Nota importante:** Para ver el efecto puro de `top_p`, se mantiene `temperature=1.0`.

In [11]:
PROMPT_TOP_P = "Describe en 3 frases las sensaciones de caminar por un bosque en otoño."

valores_top_p = [0.1, 0.5, 0.9, 1.0]
resultados_top_p = {}

for tp in valores_top_p:
    respuesta = chat(
        PROMPT_TOP_P,
        temperature=1.0,   # fijo para ver efecto puro de top_p
        top_p=tp
    )
    resultados_top_p[f"top_p={tp} (temp=1.0)"] = respuesta

mostrar_comparativa(resultados_top_p, "Efecto de top_p (temperatura fija en 1.0)")


 Efecto de top_p (temperatura fija en 1.0)

--- top_p=0.1 (temp=1.0) ---
Caminar por un bosque en otoño es como adentrarse en un cuadro vivo, donde los tonos cálidos de las hojas caídas crujen bajo tus pies con cada paso. El aire fresco y ligeramente húmedo acaricia tu rostro, impregnado del aroma terroso de la madera y las hojas en descomposición. La luz suave del sol se filtra entre las ramas desnudas, creando un ambiente sereno que invita a la introspección y la calma.


--- top_p=0.5 (temp=1.0) ---
Caminar por un bosque en otoño es como adentrarse en un cuadro vivo, donde los tonos cálidos de las hojas caídas crujen bajo tus pies con cada paso. El aire fresco y ligeramente húmedo acaricia tu rostro, impregnado del aroma terroso de la madera y las hojas en descomposición. La luz suave del sol se filtra entre las ramas desnudas, creando un ambiente sereno que invita a la introspección y la calma.


--- top_p=0.9 (temp=1.0) ---
Caminar por un bosque en otoño envuelve los sentidos con

In [12]:
# Comparación directa: top_p bajo vs alto en vocabulario usado
PROMPT_VOCAB = "Describe el sabor de una manzana con palabras muy variadas e inusuales."

print("top_p=0.1 (vocabulario restringido):")
print(chat(PROMPT_VOCAB, temperature=1.0, top_p=0.1))

print("\ntop_p=1.0 (vocabulario amplio):")
print(chat(PROMPT_VOCAB, temperature=1.0, top_p=1.0))

top_p=0.1 (vocabulario restringido):
El sabor de una manzana puede describirse como un estallido jugoso de frescura crujiente, con un toque de dulzura melosa que acaricia el paladar como un rayo de sol en un día frío. Su acidez chispeante, a veces efervescente, recuerda a un susurro de electricidad en la lengua, mientras que su textura firme y acuosa evoca la sensación de morder un fragmento de rocío matutino. Hay un eco de caramelo sutil, como un recuerdo lejano de algo cálido y reconfortante, y una nota herbácea que susurra historias de la tierra y el viento. En cada mordisco, la manzana es un equilibrio entre lo vibrante y lo sereno, como si la naturaleza hubiera encapsulado la esencia de la simplicidad y la complejidad en un solo fruto.

top_p=1.0 (vocabulario amplio):
Claro, aquí tienes una descripción del sabor de una manzana con palabras variadas e inesperadas:

El sabor de una manzana es una sinfonía crujiente y chispeante, como una caricia brillante de jugosidad que desciende 

---
## 4. Experimentación con Penalties

### ¿Qué controlan los penalties?

- **`presence_penalty`** (`-2.0` a `2.0`): Penaliza un token si **ya apareció al menos una vez** en la respuesta, independientemente de cuántas veces. Fomenta hablar de **temas nuevos**.

- **`frequency_penalty`** (`-2.0` a `2.0`): Penaliza un token proporcionalmente a **cuántas veces** ha aparecido. Cuanto más se repite, más se penaliza. Reduce repetición de **palabras concretas**.

### 4.1 — Experimento con `presence_penalty`

In [13]:
# Prompt que tiende a repetir contenido
PROMPT_PRODUCTO = "Describe detalladamente nuestro nuevo smartphone. Menciona sus características, ventajas y por qué destacamos frente a la competencia. Escribe al menos 200 palabras."

print("=== presence_penalty=0.0 (sin penalización) ===")
resp_pp0 = chat(PROMPT_PRODUCTO, temperature=0.8, presence_penalty=0.0)
print(resp_pp0)

print("\n=== presence_penalty=0.6 (penalización moderada) ===")
resp_pp06 = chat(PROMPT_PRODUCTO, temperature=0.8, presence_penalty=0.6)
print(resp_pp06)

print("\n=== presence_penalty=1.5 (penalización alta) ===")
resp_pp15 = chat(PROMPT_PRODUCTO, temperature=0.8, presence_penalty=1.5)
print(resp_pp15)

=== presence_penalty=0.0 (sin penalización) ===
¡Con gusto! Aquí tienes una descripción detallada de nuestro nuevo smartphone:

El **[Nombre del Smartphone]** es una poderosa combinación de diseño innovador, tecnología de vanguardia y funcionalidad excepcional, redefiniendo lo que significa tener un dispositivo móvil premium. Este smartphone cuenta con una pantalla AMOLED de 6.7 pulgadas con resolución 2K y una tasa de refresco de 120 Hz, lo que garantiza colores vibrantes, negros profundos y una experiencia fluida en cada interacción, ideal tanto para entretenimiento como productividad.

En su interior, está impulsado por el último procesador [Nombre del Chip] de [Fabricante], con arquitectura de 4 nanómetros que ofrece un rendimiento ultrarrápido y eficiencia energética sin precedentes. Está acompañado de hasta 12 GB de RAM y 512 GB de almacenamiento interno, para un rendimiento multitarea impecable y amplio espacio para tus archivos.

La cámara es otra joya de este dispositivo: un s

### 4.2 — Experimento con `frequency_penalty`

In [14]:
# Prompt que tiende a repetir palabras específicas
PROMPT_IDEAS = "Dame 10 ideas de negocio relacionadas con la inteligencia artificial para pequeñas empresas."

print("=== frequency_penalty=0.0 (sin penalización) ===")
resp_fp0 = chat(PROMPT_IDEAS, temperature=0.8, frequency_penalty=0.0)
print(resp_fp0)

print("\n=== frequency_penalty=0.8 (penalización alta) ===")
resp_fp08 = chat(PROMPT_IDEAS, temperature=0.8, frequency_penalty=0.8)
print(resp_fp08)

=== frequency_penalty=0.0 (sin penalización) ===
¡Claro! Aquí tienes 10 ideas de negocio relacionadas con la inteligencia artificial (IA) que pueden ser llevadas a cabo por pequeñas empresas:

### 1. **Desarrollo de chatbots personalizados**
   - Diseña e implementa chatbots para empresas locales o pequeñas empresas que necesitan mejorar su atención al cliente, automatizar respuestas frecuentes y brindar soporte 24/7.

### 2. **Servicios de análisis de datos inteligente**
   - Ofrece servicios de análisis de datos impulsados por IA para ayudar a pequeñas empresas a identificar tendencias, optimizar procesos y mejorar la toma de decisiones basadas en datos.

### 3. **Marketing digital con IA**
   - Crea una agencia de marketing que utilice herramientas de IA para optimizar campañas publicitarias, segmentar audiencias, generar contenido relevante y aumentar el retorno de inversión en publicidad.

### 4. **Plataforma de educación personalizada**
   - Desarrolla una plataforma educativa qu

### 4.3 — Combinación de ambos penalties

In [15]:
PROMPT_COMBINADO = "Escribe un párrafo describiendo los beneficios de hacer ejercicio regularmente."

configuraciones = [
    {"nombre": "Sin penalties",              "pp": 0.0, "fp": 0.0},
    {"nombre": "Solo presence_penalty=0.6",  "pp": 0.6, "fp": 0.0},
    {"nombre": "Solo frequency_penalty=0.8", "pp": 0.0, "fp": 0.8},
    {"nombre": "Ambos combinados",           "pp": 0.6, "fp": 0.8},
]

for config in configuraciones:
    print(f"\n=== {config['nombre']} (pp={config['pp']}, fp={config['fp']}) ===")
    resp = chat(
        PROMPT_COMBINADO,
        temperature=0.8,
        presence_penalty=config["pp"],
        frequency_penalty=config["fp"]
    )
    print(resp)


=== Sin penalties (pp=0.0, fp=0.0) ===
Hacer ejercicio regularmente ofrece innumerables beneficios para la salud física, mental y emocional. En el aspecto físico, mejora la condición cardiovascular, fortalece los músculos y huesos, aumenta la flexibilidad y ayuda a mantener un peso saludable, reduciendo el riesgo de enfermedades crónicas como la diabetes tipo 2, hipertensión y enfermedades del corazón. Además, tiene un impacto positivo en la salud mental, ya que libera endorfinas, conocidas como las hormonas de la felicidad, que ayudan a reducir el estrés, la ansiedad y los síntomas de la depresión. También mejora la calidad del sueño, favorece la concentración y eleva los niveles de energía durante el día. Por último, el ejercicio contribuye al bienestar general y fomenta hábitos saludables, mejorando la autoestima y promoviendo una sensación de logro y equilibrio en la vida diaria.

=== Solo presence_penalty=0.6 (pp=0.6, fp=0.0) ===
Hacer ejercicio regularmente ofrece una amplia var

---
## 5. Preguntas Teóricas

> Responde con tus propias palabras basándote en lo que observaste en los experimentos.

### Pregunta 1: ¿Cuál es la diferencia entre `top_p` y `temperature`?

> Ambos controlan la “creatividad” del modelo, pero lo hacen de forma distinta:

`temperature` → cambia cómo de “arriesgado” es el modelo al elegir palabras.
- Baja (0.1–0.3): respuestas muy predecibles, conservadoras
- Alta (0.8–1+): más creatividad, más probabilidad de decir cosas raras

 `top_p` (nucleus sampling) → recorta el conjunto de palabras posibles.
- Ej: top_p = 0.9 → solo considera el 90% de probabilidad acumulada más alta
- Es más “selectivo” que temperature

> Resumen rápido:
>
> - **temperature** = cómo decide
> - **top_p** = entre qué opciones puede decidir

### Pregunta 2: ¿Por qué no se recomienda ajustar `top_p` y `temperature` al mismo tiempo?

> Porque estás tocando dos controles que afectan a lo mismo, pero de formas diferentes. Acabarías perdiendo control, porque no dejas de estar usando dos filtros de aleatoriedad distintos a la vez. Tendrías un comportamiento impredecible y sería complicado debuggear, porque sería complicado entender qué causó el resultado.


Como buena práctica, habría que usar uno u otro dependiendo de lo que prefieras para el proyecto:
- `temperature` para control general
- `top_p` si quieres algo más acotado o controlado


### Pregunta 3: ¿Cuál es la diferencia entre `presence_penalty` y `frequency_penalty`?

> Ambos evitan repeticiones, pero no igual:

`frequency_penalty` → penaliza repetir palabras muchas veces
- Si dices “IA IA IA IA”, cada repetición pesa más
- Evita loops o redundancia

`presence_penalty` → penaliza usar cualquier palabra ya usada, aunque sea una vez
- Fuerza a introducir conceptos nuevos
- Hace el texto más variado


> Resumen rápido:
>
> - **frequency_penalty** = menos repetir lo mismo
> - **presence_penalty** = más variedad de ideas

---
## 6. Conclusiones



De todos los parámetros explorados, **temperature** es probablemente el más transversal y útil en el día a día. Básicamente porque con un solo valor puedes cambiar totalmente cómo se comporta el modelo: desde algo súper determinista hasta algo mucho más creativo. Para tareas técnicas como generación de código o extracción de datos, temperatura baja y cpara contenido creativo o conversacional, valores más altos. Muy útil y fácil de usar.

**frequency_penalty** me parece especialmente interesante para tareas de creación de contenido, por ejemplo textos para una web. En ese contexto, evitar que el modelo repita las mismas palabras una y otra vez imagino que puede llegar a ser un diferenciador real de calidad. Si los textos suenan más naturales y elaborados, un lector lo notará y apreciará más que un texto hecho por IA de forma básica. Combinado con una temperatura media, puede producir contenido bastante más rico que con configuración por defecto.

**¿Qué configuración usaría para cada tipo de tarea?** 

| Tarea | temperature | top_p | presence_penalty | frequency_penalty |
|-------|------------|-------|-----------------|-------------------|
| Generación de código | 0 - 0.3 | 0.8 - 1 | 0.0 | 0.0 |
| Extracción de datos / JSON | 0.0 | 1.0 | 0.0 | 0.0 |
| Chatbot de soporte | 0.7 | 0.7 | 0.0 | 0.3 |
| Redacción de contenido web | 0.8 | 0.8 | 0.3 | 0.6 |
| Brainstorming de ideas | 1.0 | 1 | 0.6 | 0.4 |
| Contenido creativo libre | 1.0 | 1 | 0.3 | 0.5 |